In [2]:
from importlib import import_module
from pathlib import Path
import json
import random
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / '.data/extracted').is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Open this notebook from inside the project.')
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

evaluation = import_module('src.steps.04_evaluation')
EvaluationItem = evaluation.EvaluationItem
Evaluator = evaluation.Evaluator
hf_inference = evaluation.hf_inference
ollama_inference = evaluation.ollama_inference
openrouter_inference = evaluation.openai_inference
report_results = evaluation.report_results

SAMPLE_SIZE, SEED, WORKERS = 20, 42, 1


In [3]:
def load_items(limit=None, seed=42):
    test_file = PROJECT_ROOT / '.data/curated/test.jsonl'
    if not test_file.is_file():
        raise FileNotFoundError('Run the data-curation notebook first to create .data/curated/test.jsonl.')
    with test_file.open(encoding='utf-8') as handle:
        records = [json.loads(line) for line in handle if line.strip()]
    available = len(records)
    if limit is not None:
        if limit < 1:
            raise ValueError('limit must be positive or None.')
        records = random.Random(seed).sample(records, min(limit, available))

    items = []
    for record in records:
        items.append(EvaluationItem(
            resume_id=record['resume_id'],
            category=record['category'],
            resume_text=record['resume_text'],
            target_json=record['target_json'],
        ))
    return items, available

In [4]:
items, available_pairs = load_items(SAMPLE_SIZE, SEED)
print(f'Available held-out test resumes: {available_pairs:,}')
print(f'Loaded for this test: {len(items)}')

Available held-out test resumes: 1,463
Loaded for this test: 20


In [5]:
evaluator = Evaluator(items)
reports = {}

## Optional: Hugging Face model evaluation

Use this section only if you also want to evaluate local Transformers models from the Hub.

In [ ]:
HUGGINGFACE_MODELS = [
    'google/gemma-3-270m-it',
    'HuggingFaceTB/SmolLM2-360M-Instruct',
]

for model_name in HUGGINGFACE_MODELS:
    reports[f'huggingface:{model_name}'] = evaluator.evaluate(
        callback=hf_inference(model_name),
        workers=WORKERS,
    )


## Ollama — local model evaluation

Set the model names to models already available in your local Ollama installation, then run this section.

In [ ]:
OLLAMA_MODELS = ['llama3.2', 'gemma3:270m']
for model_name in OLLAMA_MODELS:
    reports[f'ollama:{model_name}'] = evaluator.evaluate(
        callback=ollama_inference(model_name),
        workers=WORKERS,
    )


## OpenRouter — hosted model evaluation

Set `OPENROUTER_API_KEY` in `.env`, then choose OpenRouter model IDs and run this section.

In [6]:
OPENROUTER_MODELS = [
    'openai/gpt-6-luna',
]

for model_name in OPENROUTER_MODELS:
    reports[f'openrouter:{model_name}'] = evaluator.evaluate(
        callback=openrouter_inference(model_name),
        workers=WORKERS,
    )

Evaluating resumes (1 workers): 100%|██████████| 20/20 [04:41<00:00, 14.07s/resume]


## Combined comparison

Run this after the provider sections you want to evaluate to plot and compare their reports.

In [ ]:
# Display model charts and the metrics dashboard (nothing is saved yet).
summary = report_results(reports)


In [ ]:
# Save charts and metrics JSON under ./results/ when you are ready.
saved_to = summary.save()
saved_to